In [1]:
%matplotlib inline

from pathlib import Path
import heapq
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

import dataset_generator as gen

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "simulation"
DATA_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

print("Ready! Using generator:", Path(gen.__file__).resolve())
print("Dataset output folder:", DATA_DIR.resolve())

Ready! Using generator: /Users/aayush/Desktop/DSA/Graph_Routing_Project/dataset_generator.py
Dataset output folder: /Users/aayush/Desktop/DSA/Graph_Routing_Project/simulation


In [2]:
def astar_reference(ds, query=0):
    """A* on A* datasets: return (distance, path, expanded_node_ids)."""
    coords = ds["coords"]
    s, goal, _ = ds["queries"][query]
    adjacency = {u: [] for u in coords}
    for u, v, w in ds["edges"]:
        adjacency.setdefault(u, []).append((v, w))

    if ds.get("kind") == "grid":
        def h(u):
            return abs(coords[u][0] - coords[goal][0]) + abs(coords[u][1] - coords[goal][1])
    else:
        def h(u):
            return math.dist(coords[u], coords[goal])

    g = {s: 0.0}
    parent = {}
    expanded = []
    visited = set()
    pq = [(h(s), -0.0, s)]   # larger g wins ties in f

    while pq:
        _, _, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        expanded.append(u)
        if u == goal:
            path = [goal]
            while path[-1] != s:
                path.append(parent[path[-1]])
            path.reverse()
            return g[goal], path, expanded
        for v, weight in adjacency.get(u, []):
            candidate = g[u] + weight
            if candidate < g.get(v, math.inf):
                g[v] = candidate
                parent[v] = u
                heapq.heappush(pq, (candidate + h(v), -candidate, v))

    return math.inf, [], expanded


def negative_cycle_nodes(n, edges):
    """Find one negative cycle using Bellman-Ford, if any."""
    dist = [0.0] * n
    parent = [-1] * n
    last_changed = -1
    for _ in range(n):
        last_changed = -1
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
                last_changed = v
        if last_changed == -1:
            return []
    x = last_changed
    for _ in range(n):
        x = parent[x]
    backward = [x]
    y = parent[x]
    while y != x:
        backward.append(y)
        y = parent[y]
    return backward[::-1]


def pretty_distance(d):
    return "unreachable (∞)" if math.isinf(d) else f"{d:.4f}".rstrip("0").rstrip(".")

In [3]:
def plot_johnson(ds):
    n = len(ds["coords"])
    edges = ds["edges"]
    expected = ds.get("expected")
    show_graph = n <= 30
    show_matrix = n <= 200
    show_expected = isinstance(expected, list) and n <= 50
    count = int(show_graph) + 1 + int(show_expected)
    fig, axes = plt.subplots(1, count, figsize=(6.1 * count, 6), squeeze=False,
                             layout="constrained")
    axes = iter(axes[0])
    cycle = negative_cycle_nodes(n, edges) if n <= 400 else []
    cycle_edges = ({(cycle[i], cycle[(i + 1) % len(cycle)])
                    for i in range(len(cycle))} if cycle else set())

    if show_graph:
        ax = next(axes)
        angle = {i: 2 * math.pi * i / n for i in range(n)}
        pos = {i: (math.cos(angle[i]), math.sin(angle[i])) for i in range(n)}
        for u, v, w in edges:
            is_cycle = (u, v) in cycle_edges
            color = "darkorange" if is_cycle else ("crimson" if w < 0 else "#8993a2")
            ax.annotate("", xy=pos[v], xytext=pos[u],
                        arrowprops=dict(arrowstyle="-|>", color=color,
                                        linewidth=2 if is_cycle else (1.35 if w < 0 else .65),
                                        alpha=0.85, shrinkA=13, shrinkB=13,
                                        connectionstyle="arc3,rad=0.12"))
            if n <= 8:
                x1, y1 = pos[u]
                x2, y2 = pos[v]
                ax.text((x1 + x2) / 2, (y1 + y2) / 2, f"{w:g}",
                        ha="center", va="center", fontsize=8,
                        bbox={"facecolor": "white", "edgecolor": "none", "alpha": .8})
        for u, (x, y) in pos.items():
            ax.scatter(x, y, color="#2563eb", s=400, zorder=4)
            ax.text(x, y, str(u), color="white", weight="bold", fontsize=9,
                    ha="center", va="center", zorder=5)
        ax.axis("off")
        ax.set_aspect("equal")
        ax.set_xlim(-1.3, 1.3)
        ax.set_ylim(-1.3, 1.3)
        ax.set_title("Graph: red = negative edge; orange = negative cycle")

    ax = next(axes)
    if show_matrix:
        matrix = np.full((n, n), np.nan)
        for u, v, w in edges:
            matrix[u, v] = w
        masked = np.ma.masked_invalid(matrix)
        cmap = plt.get_cmap("coolwarm").copy()
        cmap.set_bad("#eeeeee")
        image = ax.imshow(masked, cmap=cmap)
        fig.colorbar(image, ax=ax, fraction=.046, label="weight")
        ax.set_xlabel("destination")
        ax.set_ylabel("source")
        ax.set_title("Edge weights (grey = no edge)")
        if n <= 10:
            for i in range(n):
                for j in range(n):
                    if not np.isnan(matrix[i, j]):
                        ax.text(j, i, f"{matrix[i,j]:g}", ha="center", va="center", fontsize=7)
    else:
        ax.hist([w for _, _, w in edges], bins=25, edgecolor="white")
        ax.set_title("Edge weights (large graph)")
        ax.set_xlabel("weight")
        ax.set_ylabel("count")

    if show_expected:
        ax = next(axes)
        distances = np.array(expected, dtype=float)
        masked = np.ma.masked_invalid(distances)
        image = ax.imshow(masked, cmap="coolwarm")
        fig.colorbar(image, ax=ax, fraction=.046, label="shortest distance")
        ax.set_title("Expected all-pairs shortest distances")
        ax.set_xlabel("destination")
        ax.set_ylabel("source")
        if n <= 10:
            for i in range(n):
                for j in range(n):
                    v = distances[i, j]
                    ax.text(j, i, "∞" if math.isinf(v) else f"{v:g}",
                            ha="center", va="center", fontsize=7)

    label = "NEGATIVE CYCLE (invalid for Johnson)" if expected == "NEGATIVE_CYCLE" else "No negative cycle expected"
    fig.suptitle(f"{ds['name']}  |  {n} nodes, {len(edges)} edges  |  {label}", fontsize=12)
    return fig


def plot_astar(ds, query=0):
    if not ds["queries"]:
        raise ValueError("This A* dataset contains no queries")
    coords = ds["coords"]
    edges = ds["edges"]
    start, goal, expected = ds["queries"][query]
    distance, path, expanded = astar_reference(ds, query)
    fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")

    if ds.get("kind") == "grid":
        width = int(max(x for x, y in coords.values())) + 1
        height = int(max(y for x, y in coords.values())) + 1
        open_cells = np.zeros((height, width))
        for x, y in coords.values():
            open_cells[int(y), int(x)] = 1
        ax.imshow(open_cells, cmap="gray", vmin=0, vmax=1, origin="upper")
        if expanded:
            ax.scatter([coords[u][0] for u in expanded],
                       [coords[u][1] for u in expanded],
                       s=8, c="orange", alpha=.55, zorder=2, label=f"Expanded ({len(expanded)})")
        ax.set_xlim(-.5, width - .5)
        ax.set_ylim(height - .5, -.5)
        ax.set_aspect("equal")
    else:
        pairs = set()
        line_segments = []
        for u, v, _ in edges:
            pair = tuple(sorted((u, v)))
            if pair not in pairs:
                pairs.add(pair)
                if len(line_segments) < 3000:
                    line_segments.append([coords[u], coords[v]])
        ax.add_collection(LineCollection(line_segments, colors="#cbd5e1",
                                         linewidths=.55, alpha=.65, zorder=1))
        ax.scatter([xy[0] for xy in coords.values()],
                   [xy[1] for xy in coords.values()],
                   s=12, c="#2563eb", zorder=2)
        if expanded:
            ax.scatter([coords[u][0] for u in expanded],
                       [coords[u][1] for u in expanded],
                       s=26, c="orange", alpha=.65, zorder=3,
                       label=f"Expanded ({len(expanded)})")
        ax.autoscale()
        ax.invert_yaxis()
        ax.set_aspect("equal", adjustable="datalim")

    if path:
        ax.plot([coords[u][0] for u in path], [coords[u][1] for u in path],
                color="crimson", linewidth=2.6, zorder=5,
                label=f"Shortest path ({pretty_distance(distance)})")
    else:
        ax.text(.5, .5, "Goal unreachable", transform=ax.transAxes,
                fontsize=14, weight="bold", color="crimson", ha="center",
                bbox=dict(facecolor="white", alpha=.85, edgecolor="none"))

    ax.scatter(*coords[start], s=155, color="limegreen", edgecolor="black",
               zorder=6, label=f"Start ({start})")
    ax.scatter(*coords[goal], s=180, color="gold", edgecolor="black",
               marker="*", zorder=7, label=f"Goal ({goal})")
    ax.set_title(f"{ds['name']} — A* query {query + 1}/{len(ds['queries'])}")
    ax.legend(loc="best", fontsize=8)

    matches = ((math.isinf(distance) and math.isinf(expected)) or
               math.isclose(distance, expected, rel_tol=1e-8, abs_tol=1e-6))
    print("A* expected:", pretty_distance(expected),
          "| computed:", pretty_distance(distance),
          "| expanded:", len(expanded),
          "| reference check:", "PASS" if matches else "MISMATCH")
    return fig


def display_dataset(ds, query=0, save_png=True, out_dir=None):
    """Draw Johnson or A* and optionally save the PNG beside its dataset."""
    if ds["mode"] == "johnson":
        fig = plot_johnson(ds)
    elif ds["mode"] == "astar":
        fig = plot_astar(ds, query)
    else:
        raise ValueError(f"Unknown dataset mode: {ds['mode']}")
    if save_png:
        folder = Path(out_dir) if out_dir is not None else DATA_DIR / ds["mode"]
        image_dir = folder / "images"
        image_dir.mkdir(parents=True, exist_ok=True)
        suffix = f"_query{query + 1}" if ds["mode"] == "astar" else ""
        dest = image_dir / f"{ds['name']}{suffix}.png"
        fig.savefig(dest, dpi=140, bbox_inches="tight")
        print("Image saved:", dest)
    plt.show()
    plt.close(fig)
    return fig

In [4]:
def ask_int(label, default, low=None, high=None):
    while True:
        value = input(f"{label} [default {default}]: ").strip()
        try:
            num = default if not value else int(value)
        except ValueError:
            print("Please enter a whole number.")
            continue
        if (low is not None and num < low) or (high is not None and num > high):
            print(f"Enter a value from {low} to {high}.")
            continue
        return num


def ask_float(label, default, low=None, high=None):
    while True:
        value = input(f"{label} [default {default}]: ").strip()
        try:
            num = default if not value else float(value)
        except ValueError:
            print("Please enter a number.")
            continue
        if (low is not None and num < low) or (high is not None and num > high):
            print(f"Enter a value from {low} to {high}.")
            continue
        return num


def ask_yes(label, default=True):
    while True:
        value = input(f"{label} [{'Y/n' if default else 'y/N'}]: ").strip().lower()
        if not value:
            return default
        if value in ("yes", "y"):
            return True
        if value in ("no", "n"):
            return False
        print("Please answer y or n.")


def ask_seed():
    while True:
        raw = input("Random seed [blank = NEW random graph]: ").strip()
        if not raw:
            return None
        try:
            return int(raw)
        except ValueError:
            print("Enter an integer seed or leave blank.")


def ask_dataset():
    """Build one new dataset from prompts, or select a saved dataset."""
    print("\n========== GRAPH SIMULATION ==========")
    print("1) Johnson — random weighted graph")
    print("2) Johnson — WITH a negative cycle")
    print("3) A* — grid with obstacles")
    print("4) A* — random geometric graph")
    print("5) Johnson — professor's fixed test cases")
    print("6) Open a previously saved dataset")
    print("0) Exit")
    choice = input("Choose an option [0-6]: ").strip()

    if choice in ("0", "q", "quit", ""):
        return "EXIT", None

    if choice in ("1", "2"):
        n = ask_int("Number of vertices", 10, 2, 5000)
        max_edges = n * (n - 1)
        edges = ask_int("Number of directed edges", min(30, max_edges), 1, max_edges)
        connected = edges >= n and ask_yes("Force strongly connected?", True)
        ds = gen.johnson_dataset(n=n, edge_count=edges, seed=ask_seed(),
                                 connected=connected, negative_cycle=(choice == "2"))
        return "NEW", ds

    if choice == "3":
        rows = ask_int("Grid rows", 20, 2, 1000)
        cols = ask_int("Grid columns", 20, 2, 1000)
        prob = ask_float("Obstacle probability (0.0-0.6)", 0.2, 0.0, 0.6)
        start_r = ask_int("Start row", 0, 0, rows - 1)
        start_c = ask_int("Start column", 0, 0, cols - 1)
        goal_r = ask_int("Goal row", rows - 1, 0, rows - 1)
        goal_c = ask_int("Goal column", cols - 1, 0, cols - 1)
        ds = gen.grid_dataset(rows=rows, cols=cols, obstacle_prob=prob,
                              seed=ask_seed(), start=(start_r, start_c),
                              goal=(goal_r, goal_c))
        return "NEW", ds

    if choice == "4":
        n = ask_int("Number of vertices", 60, 3, 5000)
        isolate = ask_yes("Isolate last node for unreachable-path test?", False)
        m = n - 1 if isolate else n
        max_extra = m * (m - 1) // 2 - (m - 1)
        extra = ask_int("Extra undirected edges beyond spanning tree",
                        min(100, max_extra), 0, max_extra)
        queries = ask_int("Number of random start-goal queries", 3, 1, 50)
        ds = gen.geometric_dataset(n=n, extra_edges=extra, seed=ask_seed(),
                                   isolate_last=isolate, num_queries=queries)
        return "NEW", ds

    if choice == "5":
        cases = gen.fixed_cases()
        ix = ask_int("Professor case (1=negative weights, 2=negative cycle)", 1, 1, 2)
        return "NEW", cases[ix - 1]

    if choice == "6":
        candidates = sorted(DATA_DIR.rglob("*.txt"), key=lambda p: p.stat().st_mtime, reverse=True)
        if not candidates:
            print("No saved datasets found. Generate one first.")
            return "SKIP", None
        for ix, path in enumerate(candidates, 1):
            print(f"{ix:>3}) {path.relative_to(BASE_DIR)}")
        ix = ask_int("Which dataset number?", 1, 1, len(candidates))
        path = candidates[ix - 1]
        return "LOADED", (gen.load_dataset(str(path)), path.parent)

    print("Unknown choice. Choose 0-6.")
    return "SKIP", None


def run_simulation():
    while True:
        try:
            action, payload = ask_dataset()
            if action == "EXIT":
                print("Simulation menu closed.")
                break
            if action == "SKIP":
                continue
            if action == "LOADED":
                ds, output_dir = payload
            else:
                ds = payload
                output_dir = DATA_DIR / ds["mode"]
                output_path = gen.save_dataset(ds, str(output_dir))
                print("Dataset saved:", output_path)

            print(gen.describe(ds))
            q = 0
            if ds["mode"] == "astar" and len(ds["queries"]) > 1:
                for i, (s, t, d) in enumerate(ds["queries"], 1):
                    print(f"  Query {i}: {s} -> {t}; expected {pretty_distance(d)}")
                q = ask_int("Which A* query to visualize?", 1, 1, len(ds["queries"])) - 1
            display_dataset(ds, query=q, save_png=True, out_dir=output_dir)
        except (ValueError, OSError, AssertionError) as exc:
            print("Error:", exc)
            print("Choose an option again.")

In [5]:
run_simulation()


========== GRAPH SIMULATION ==========
1) Johnson — random weighted graph
2) Johnson — WITH a negative cycle
3) A* — grid with obstacles
4) A* — random geometric graph
5) Johnson — professor's fixed test cases
6) Open a previously saved dataset
0) Exit
Simulation menu closed.
